# 3D porous Darcy convection: dataset generation

Data-only extraction from `GRNO_PorousConvection3D_Ra1000_64 (3).ipynb`. Numerical solver implementations, physical parameters, split seeds, and original profile defaults are preserved. Neural models, training, checkpoint evaluation, and model-result plotting are omitted. Source cell numbers are recorded in each code cell's metadata.

The saved default is `PROFILE = "paper"`. Read the profile table in [README.md](README.md) before running: the `paper` profile can require substantial GPU time and disk space. Install a compatible PyTorch build plus NumPy, pandas, tqdm, and IPython in this notebook's kernel. These notebooks do not install packages automatically.

Run from the repository, or set the environment variable `GRNO_ROOT` to your intended project/data location before starting the kernel. The printed `ROOT` and data roots identify the generated files. Restart and run from the top after changing profiles. Set `RUN_DATA_GENERATION = False` to inspect a complete existing cache; simulator audits and normalization still run.

The original `paper` profile simulates on a 128³ reference grid and volume-averages to stored 64³ concentration arrays. It keeps 101 frames per trajectory in every split. The original transform, boundary/flux preflight, generation diagnostics, normalization, and dataset-readiness gate are retained. The optional later spatial/temporal reference-grid audit cell (original cell 29) is outside this compact extraction; this release has not executed that optional audit. Original legacy `RunConfig` fields supply the readiness audit horizon and fixed evaluation starts; no model is instantiated.


In [ ]:
from __future__ import annotations

import contextlib
import dataclasses
import hashlib
import json
import math
import os
import random
import shutil
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from IPython.display import display
from tqdm.auto import tqdm

# Storage location only: set GRNO_ROOT before starting this kernel to override it.
# No solver, split, seed, or physical parameter is changed by this root selection.
def locate_project_root() -> Path:
    override = os.environ.get("GRNO_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    return cwd

ROOT = locate_project_root()
ROOT.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print({"project_root": str(ROOT), "device": str(device), "torch": torch.__version__})


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def short_hash(payload) -> str:
    raw = json.dumps(payload, sort_keys=True, default=str).encode()
    return hashlib.sha256(raw).hexdigest()[:12]


def atomic_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".partial")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str))
    os.replace(tmp, path)


## Original profiles and physical parameters


In [ ]:
PROFILE = "paper"                 # "smoke", "pilot", or "paper"


RUN_DATA_GENERATION = True


ALLOW_WEAK_DATASET = False       # never silently publish a near-persistence task


DATASET_VERSION = "porous-convection3d-fv-strang-rk2-continuum-ic-v2"


STORAGE_NUMPY_DTYPE = "float32"


STORAGE_BYTES = np.dtype(STORAGE_NUMPY_DTYPE).itemsize


@dataclass(frozen=True)
class RunConfig:
    profile: str
    epochs: int
    samples_per_epoch: int
    micro_batch: int
    grad_accum: int
    train_rollout: int
    val_rollout: int
    test_rollout: int
    width: int
    depths: tuple
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_weight: float = 0.05
    spectral_weight: float = 0.02
    rollout_decay: float = 0.8
    use_amp: bool = True

    @property
    def effective_batch(self):
        return self.micro_batch * self.grad_accum

    @property
    def optimizer_updates_per_epoch(self):
        if self.samples_per_epoch % self.micro_batch:
            raise ValueError("samples_per_epoch must be divisible by micro_batch")
        microbatches = self.samples_per_epoch // self.micro_batch
        if microbatches % self.grad_accum:
            raise ValueError("microbatches_per_epoch must divide grad_accum")
        return microbatches // self.grad_accum

    @property
    def microbatches_per_epoch(self):
        return self.samples_per_epoch // self.micro_batch


@dataclass(frozen=True)
class PorousConfig:
    reference_n: int
    model_n: int
    length_x: float
    length_y: float
    depth: float
    rayleigh_darcy: float
    frame_dt: float
    max_solver_dt: float
    advective_cfl: float
    initial_time: float
    perturbation: float
    perturbation_max_mode: int
    warmup_time: float
    n_train: int
    n_val: int
    n_test: int
    train_transitions: int
    val_transitions: int
    test_transitions: int
    generation_batch: int

    @property
    def diffusivity(self):
        return 1.0 / self.rayleigh_darcy

    @property
    def downsample_factor(self):
        if self.reference_n % self.model_n:
            raise ValueError("reference_n must be divisible by model_n")
        return self.reference_n // self.model_n


if PROFILE == "smoke":
    RUN = RunConfig(PROFILE, epochs=2, samples_per_epoch=8, micro_batch=1,
                    grad_accum=2, train_rollout=2, val_rollout=3,
                    test_rollout=5, width=8, depths=(1, 1, 1, 1),
                    lr=1e-3, spectral_weight=0.0, use_amp=False)
    PDE = PorousConfig(
        reference_n=16, model_n=16, length_x=1.0, length_y=1.0, depth=1.0,
        rayleigh_darcy=100.0, frame_dt=0.04, max_solver_dt=0.02,
        advective_cfl=0.35, initial_time=0.40, perturbation=0.03,
        perturbation_max_mode=3, warmup_time=0.04,
        n_train=2, n_val=1, n_test=1,
        train_transitions=6, val_transitions=8, test_transitions=10,
        generation_batch=1)
elif PROFILE == "pilot":
    RUN = RunConfig(PROFILE, epochs=20, samples_per_epoch=128, micro_batch=1,
                    grad_accum=4, train_rollout=4, val_rollout=10,
                    test_rollout=40, width=16, depths=(1, 1, 2, 2))
    PDE = PorousConfig(
        reference_n=64, model_n=64, length_x=1.0, length_y=1.0, depth=1.0,
        rayleigh_darcy=1000.0, frame_dt=0.10, max_solver_dt=0.02,
        advective_cfl=0.35, initial_time=1.00, perturbation=0.02,
        perturbation_max_mode=8, warmup_time=0.20,
        n_train=8, n_val=2, n_test=2,
        train_transitions=50, val_transitions=80, test_transitions=80,
        generation_batch=2)
elif PROFILE == "paper":
    RUN = RunConfig(PROFILE, epochs=100, samples_per_epoch=512, micro_batch=1,
                    grad_accum=4, train_rollout=4, val_rollout=10,
                    test_rollout=40, width=24, depths=(2, 2, 3, 4))
    PDE = PorousConfig(
        reference_n=128, model_n=64, length_x=1.0, length_y=1.0, depth=1.0,
        rayleigh_darcy=1000.0, frame_dt=0.10, max_solver_dt=0.02,
        advective_cfl=0.35, initial_time=1.00, perturbation=0.02,
        perturbation_max_mode=8, warmup_time=0.20,
        n_train=64, n_val=8, n_test=12,
        train_transitions=100, val_transitions=100, test_transitions=100,
        generation_batch=2)
else:
    raise ValueError(f"Unknown PROFILE={PROFILE!r}")


if PROFILE != "smoke" and device.type != "cuda":
    raise RuntimeError("pilot/paper profiles require CUDA; use smoke for CPU checks")


DATA_SIGNATURE = short_hash({
    "dataset_version": DATASET_VERSION,
    "pde": asdict(PDE),
    "storage_numpy_dtype": STORAGE_NUMPY_DTYPE,
})


DATA_ROOT = (
    ROOT / "data" / "porous_convection3d" / PROFILE / DATA_SIGNATURE)


estimated_gb = (
    (PDE.n_train * (PDE.train_transitions + 1)
     + PDE.n_val * (PDE.val_transitions + 1)
     + PDE.n_test * (PDE.test_transitions + 1))
    * PDE.model_n ** 3 * STORAGE_BYTES / 1e9
)


disk = shutil.disk_usage(ROOT)


if disk.free / 1e9 < estimated_gb + 3:
    raise RuntimeError("Not enough free disk for data plus checkpoint headroom")


DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(json.dumps({"profile": PROFILE, "pde": asdict(PDE), "data_signature": DATA_SIGNATURE, "estimated_dataset_GB": estimated_gb, "data_root": str(DATA_ROOT)}, indent=2))


## Original slab Green map and boundary-aware diffusion


In [ ]:
def dct2_complex(field):
    """Unnormalised DCT-II along axis 1, including complex-valued fields."""
    n = field.shape[1]
    extended = torch.cat((field, torch.flip(field, dims=(1,))), dim=1)
    spectrum = torch.fft.fft(extended, dim=1)
    k = torch.arange(n, device=field.device, dtype=torch.float32)
    phase = torch.exp(
        -0.5j * math.pi * k / n).reshape(1, n, 1, 1)
    return 0.5 * spectrum[:, :n] * phase


def idct2_complex(coefficients):
    """Inverse of dct2_complex along axis 1."""
    n = coefficients.shape[1]
    k = torch.arange(n, device=coefficients.device, dtype=torch.float32)
    phase = torch.exp(
        0.5j * math.pi * k / n).reshape(1, n, 1, 1)
    spectrum = coefficients.new_zeros(
        coefficients.shape[0], 2 * n,
        coefficients.shape[2], coefficients.shape[3])
    spectrum[:, :n] = 2.0 * coefficients * phase
    if n > 1:
        reverse_k = torch.arange(
            n - 1, 0, -1,
            device=coefficients.device, dtype=torch.float32)
        reverse_phase = torch.exp(
            -0.5j * math.pi * reverse_k / n
        ).reshape(1, n - 1, 1, 1)
        spectrum[:, n + 1:] = (
            2.0 * torch.flip(coefficients[:, 1:], dims=(1,))
            * reverse_phase
        )
    return torch.fft.ifft(spectrum, dim=1)[:, :n]


def _dst4_real(field):
    """Unnormalised real DST-IV along axis 1 via an FFT embedding."""
    n = field.shape[1]
    embedded = field.new_zeros(
        field.shape[0], 8 * n, field.shape[2], field.shape[3])
    embedded[:, 1:2 * n:2] = field
    transformed = torch.fft.fft(embedded, dim=1)
    return -transformed[:, 1:2 * n:2].imag


def dst4_complex(field):
    return torch.complex(
        _dst4_real(field.real), _dst4_real(field.imag))


def idst4_complex(coefficients):
    n = coefficients.shape[1]
    return (2.0 / n) * torch.complex(
        _dst4_real(coefficients.real),
        _dst4_real(coefficients.imag),
    )


class SlabDarcyGreen3D(nn.Module):
    """Boundary-aware discrete Green/Leray map C -> u in a periodic slab.

    x,y are periodic. z is cell-centred, with impermeable top and bottom.
    The projection uses a horizontal FFT and the exact eigendecomposition of
    the matching finite-volume Neumann Laplacian in z.
    """

    def __init__(self, n, lengths):
        super().__init__()
        self.n = int(n)
        self.depth, self.length_y, self.length_x = (float(v) for v in lengths)
        self.dz = self.depth / self.n
        self.dy = self.length_y / self.n
        self.dx = self.length_x / self.n
        kz_index = torch.arange(self.n)
        lambda_z = (
            -4.0
            * torch.sin(math.pi * kz_index / (2 * self.n)).square()
            / self.dz ** 2
        )
        ky_index = torch.fft.fftfreq(self.n) * self.n
        kx_index = torch.arange(self.n // 2 + 1)
        lambda_y = -4.0 * torch.sin(math.pi * ky_index / self.n).square() / self.dy ** 2
        lambda_x = -4.0 * torch.sin(math.pi * kx_index / self.n).square() / self.dx ** 2
        denominator = (
            lambda_z[:, None, None]
            + lambda_y[None, :, None]
            + lambda_x[None, None, :]
        )
        inverse = torch.where(
            denominator.abs() > 1e-10,
            1.0 / denominator,
            torch.zeros_like(denominator),
        )
        self.register_buffer("poisson_inverse", inverse)
        self.lengths = (self.depth, self.length_y, self.length_x)

    def _pressure(self, rhs):
        rhs_hat = torch.fft.rfft2(rhs.float(), dim=(-2, -1))
        coefficients = dct2_complex(rhs_hat)
        pressure_hat = idct2_complex(
            coefficients * self.poisson_inverse)
        return torch.fft.irfft2(
            pressure_hat, s=(self.n, self.n), dim=(-2, -1)).real

    def face_and_cell_velocity(self, concentration):
        if concentration.ndim != 5 or concentration.shape[1] != 1:
            raise ValueError("Expected concentration [B,1,D,H,W]")
        q = concentration[:, 0].float()
        b, d, h, w = q.shape
        if (d, h, w) != (self.n, self.n, self.n):
            raise ValueError(f"Expected cubic spatial shape {self.n}^3")

        buoyancy_face = q.new_zeros(b, d + 1, h, w)
        buoyancy_face[:, 1:d] = 0.5 * (q[:, :-1] + q[:, 1:])
        rhs = (buoyancy_face[:, 1:] - buoyancy_face[:, :-1]) / self.dz
        rhs_correction = rhs.mean((-3, -2, -1), keepdim=True)
        rhs = rhs - rhs_correction
        pressure = self._pressure(rhs)

        grad_x_face = (torch.roll(pressure, -1, -1) - pressure) / self.dx
        grad_y_face = (torch.roll(pressure, -1, -2) - pressure) / self.dy
        grad_z_face = q.new_zeros(b, d + 1, h, w)
        grad_z_face[:, 1:d] = (pressure[:, 1:] - pressure[:, :-1]) / self.dz

        ux_face = -grad_x_face
        uy_face = -grad_y_face
        uz_face = buoyancy_face - grad_z_face
        uz_face[:, 0].zero_()
        uz_face[:, -1].zero_()

        ux = 0.5 * (ux_face + torch.roll(ux_face, 1, -1))
        uy = 0.5 * (uy_face + torch.roll(uy_face, 1, -2))
        uz = 0.5 * (uz_face[:, :-1] + uz_face[:, 1:])
        velocity = torch.stack((ux, uy, uz), dim=1)
        return velocity, (ux_face, uy_face, uz_face), rhs_correction

    def forward(self, concentration):
        velocity, _, _ = self.face_and_cell_velocity(concentration)
        return velocity

    def face_divergence(self, faces):
        ux_face, uy_face, uz_face = faces
        return (
            (ux_face - torch.roll(ux_face, 1, -1)) / self.dx
            + (uy_face - torch.roll(uy_face, 1, -2)) / self.dy
            + (uz_face[:, 1:] - uz_face[:, :-1]) / self.dz
        )


class MixedBoundaryDiffusion3D(nn.Module):
    """Exact discrete diffusion semigroup for s=1-C.

    s has homogeneous Dirichlet data at the top face and homogeneous Neumann
    data at the bottom face.
    """

    def __init__(self, n, lengths, diffusivity):
        super().__init__()
        depth, length_y, length_x = (float(v) for v in lengths)
        dz, dy, dx = depth / n, length_y / n, length_x / n
        kz_index = torch.arange(n)
        lambda_z = (
            -4.0
            * torch.sin(
                math.pi * (kz_index + 0.5) / (2 * n)
            ).square()
            / dz ** 2
        )
        ky_index = torch.fft.fftfreq(n) * n
        kx_index = torch.arange(n // 2 + 1)
        lambda_y = -4.0 * torch.sin(math.pi * ky_index / n).square() / dy ** 2
        lambda_x = -4.0 * torch.sin(math.pi * kx_index / n).square() / dx ** 2
        spectrum = (
            lambda_z[:, None, None]
            + lambda_y[None, :, None]
            + lambda_x[None, None, :]
        )
        self.register_buffer("spectrum", spectrum)
        self.diffusivity = float(diffusivity)
        self.n = int(n)

    def forward(self, concentration, dt):
        if float(dt) == 0:
            return concentration
        deficit = 1.0 - concentration[:, 0].float()
        field_hat = torch.fft.rfft2(deficit, dim=(-2, -1))
        coefficients = dst4_complex(field_hat)
        decay = torch.exp(
            self.diffusivity * float(dt) * self.spectrum)
        out_hat = idst4_complex(coefficients * decay)
        out = torch.fft.irfft2(
            out_hat, s=(self.n, self.n), dim=(-2, -1)).real
        return (1.0 - out)[:, None]


## Original finite-volume reference integrator


In [ ]:
def minmod3(a, b, c):
    same = (torch.sign(a) == torch.sign(b)) & (torch.sign(a) == torch.sign(c))
    magnitude = torch.minimum(a.abs(), torch.minimum(b.abs(), c.abs()))
    return torch.where(same, torch.sign(a) * magnitude, torch.zeros_like(a))


def mc_periodic_slope(q, dim):
    backward = q - torch.roll(q, 1, dim)
    forward = torch.roll(q, -1, dim) - q
    return minmod3(0.5 * (backward + forward), 2 * backward, 2 * forward)


def mc_vertical_slope(q):
    top_ghost = 2.0 - q[:, :1]
    bottom_ghost = q[:, -1:]
    extended = torch.cat((top_ghost, q, bottom_ghost), dim=1)
    backward = extended[:, 1:-1] - extended[:, :-2]
    forward = extended[:, 2:] - extended[:, 1:-1]
    return minmod3(0.5 * (backward + forward), 2 * backward, 2 * forward)


class PorousConvectionReference3D:
    """Second-order finite-volume Strang/RK2 reference generator.

    This simulator is used only to create supervision. It is never called by
    the learned model during training or inference.
    """

    def __init__(self, cfg, n=None, target_device=device):
        self.cfg = cfg
        self.n = int(n or cfg.reference_n)
        self.device = target_device
        self.lengths = (cfg.depth, cfg.length_y, cfg.length_x)
        self.dz = cfg.depth / self.n
        self.dy = cfg.length_y / self.n
        self.dx = cfg.length_x / self.n
        self.green = SlabDarcyGreen3D(self.n, self.lengths).to(target_device)
        self.diffusion = MixedBoundaryDiffusion3D(
            self.n, self.lengths, cfg.diffusivity).to(target_device)

    def advection_rhs(self, concentration, return_diagnostics=False):
        q = concentration[:, 0].float()
        velocity, faces, correction = self.green.face_and_cell_velocity(concentration)
        ux_face, uy_face, uz_face = faces

        sx = mc_periodic_slope(q, -1)
        qlx = q + 0.5 * sx
        qrx = torch.roll(q, -1, -1) - 0.5 * torch.roll(sx, -1, -1)
        flux_x = ux_face * torch.where(ux_face >= 0, qlx, qrx)

        sy = mc_periodic_slope(q, -2)
        qly = q + 0.5 * sy
        qry = torch.roll(q, -1, -2) - 0.5 * torch.roll(sy, -1, -2)
        flux_y = uy_face * torch.where(uy_face >= 0, qly, qry)

        sz = mc_vertical_slope(q)
        flux_z = q.new_zeros(q.shape[0], self.n + 1, self.n, self.n)
        qlz = q[:, :-1] + 0.5 * sz[:, :-1]
        qrz = q[:, 1:] - 0.5 * sz[:, 1:]
        interior_velocity = uz_face[:, 1:-1]
        flux_z[:, 1:-1] = interior_velocity * torch.where(
            interior_velocity >= 0, qlz, qrz)

        divergence = (
            (flux_x - torch.roll(flux_x, 1, -1)) / self.dx
            + (flux_y - torch.roll(flux_y, 1, -2)) / self.dy
            + (flux_z[:, 1:] - flux_z[:, :-1]) / self.dz
        )
        rhs = -divergence[:, None]
        if not return_diagnostics:
            return rhs
        return rhs, {
            "velocity": velocity,
            "faces": faces,
            "pressure_rhs_mean_correction": correction,
        }

    @torch.no_grad()
    def substep(self, concentration, dt):
        half = self.diffusion(concentration, 0.5 * dt)
        k1 = self.advection_rhs(half)
        stage = half + dt * k1
        k2 = self.advection_rhs(stage)
        advected = 0.5 * half + 0.5 * (stage + dt * k2)
        return self.diffusion(advected, 0.5 * dt)

    @torch.no_grad()
    def advance_frame(self, concentration, frame_dt=None):
        target = float(frame_dt if frame_dt is not None else self.cfg.frame_dt)
        elapsed, steps = 0.0, 0
        max_speed = 0.0
        integrated_top_flux = torch.zeros(
            concentration.shape[0], device=concentration.device,
            dtype=torch.float32)
        while elapsed < target - 1e-12:
            velocity, faces, _ = self.green.face_and_cell_velocity(concentration)
            ux_face, uy_face, uz_face = faces
            # The explicit MUSCL update uses face fluxes, so the CFL bound must
            # use those same face velocities rather than cell averages.
            ux = float(ux_face.abs().amax())
            uy = float(uy_face.abs().amax())
            uz = float(uz_face.abs().amax())
            advective_rate = ux / self.dx + uy / self.dy + uz / self.dz
            cfl_dt = (
                self.cfg.advective_cfl / max(advective_rate, 1e-8)
                if advective_rate > 0 else self.cfg.max_solver_dt
            )
            dt = min(self.cfg.max_solver_dt, cfl_dt, target - elapsed)
            if dt < 1e-7:
                raise RuntimeError("Reference solver time step collapsed")
            flux_before = self.top_flux(concentration)
            concentration = self.substep(concentration, dt)
            flux_after = self.top_flux(concentration)
            integrated_top_flux += (
                0.5 * dt * (flux_before + flux_after))
            elapsed += dt
            steps += 1
            max_speed = max(max_speed, ux, uy, uz)
            if steps > 10000:
                raise RuntimeError("Too many reference-solver substeps")
        return concentration, {
            "substeps": steps,
            "max_speed": max_speed,
            "integrated_top_flux": integrated_top_flux,
        }

    def sampled_initial_time(self, seed):
        generator = torch.Generator(device="cpu").manual_seed(int(seed))
        jitter = 0.9 + 0.2 * torch.rand((), generator=generator)
        return self.cfg.initial_time * float(jitter)

    @torch.no_grad()
    def initial_condition(self, seeds):
        fields = []
        z = (torch.arange(self.n, device=self.device) + 0.5) * self.dz
        y = (torch.arange(self.n, device=self.device) + 0.5) * self.dy
        x = (torch.arange(self.n, device=self.device) + 0.5) * self.dx
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        modes = []
        for ky in range(-self.cfg.perturbation_max_mode,
                        self.cfg.perturbation_max_mode + 1):
            for kx in range(-self.cfg.perturbation_max_mode,
                            self.cfg.perturbation_max_mode + 1):
                radius = math.sqrt(kx * kx + ky * ky)
                # Keep one representative of each +/- pair. The finite mode
                # set makes a trajectory seed a continuum initial condition:
                # evaluating the same coefficients on 128^3 and 192^3 grids
                # is then a genuinely matched refinement test.
                representative = ky > 0 or (ky == 0 and kx > 0)
                if representative and 1.0 <= radius <= self.cfg.perturbation_max_mode:
                    modes.append((kx, ky))
        for seed in seeds:
            generator = torch.Generator(device="cpu").manual_seed(int(seed))
            jitter = 0.9 + 0.2 * torch.rand((), generator=generator)
            t0 = self.cfg.initial_time * float(jitter)
            base = torch.erfc(
                z / (2 * math.sqrt(self.cfg.diffusivity * t0))).reshape(
                    self.n, 1, 1)
            coefficients = torch.randn(
                len(modes), 2, generator=generator,
                dtype=torch.float32).to(self.device)
            smooth = torch.zeros_like(xx)
            for mode_index, (kx, ky) in enumerate(modes):
                phase = 2 * math.pi * (
                    kx * xx / self.cfg.length_x
                    + ky * yy / self.cfg.length_y)
                smooth = (
                    smooth
                    + coefficients[mode_index, 0] * torch.cos(phase)
                    + coefficients[mode_index, 1] * torch.sin(phase)
                )
            analytic_rms = torch.sqrt(
                0.5 * coefficients.square().sum()).clamp_min(1e-7)
            smooth = smooth / analytic_rms
            envelope = 4.0 * base * (1.0 - base)
            field = base + self.cfg.perturbation * envelope * smooth[None]
            field = field.clamp(0.0, 1.0)
            fields.append(field)
        return torch.stack(fields, dim=0)[:, None]

    def top_flux(self, concentration):
        q = concentration[:, 0].float()
        gradient = 2.0 * (q[:, 0] - 1.0) / self.dz
        return -self.cfg.diffusivity * gradient.mean((-2, -1))

    def diagnostics(self, concentration):
        velocity, faces, correction = self.green.face_and_cell_velocity(concentration)
        divergence = self.green.face_divergence(faces)
        scale = velocity.square().mean().sqrt() / min(self.dx, self.dy, self.dz)
        return {
            "mass": concentration.float().mean((-3, -2, -1))[:, 0],
            "top_flux": self.top_flux(concentration),
            "min": concentration.float().amin((-3, -2, -1))[:, 0],
            "max": concentration.float().amax((-3, -2, -1))[:, 0],
            "velocity_rms": velocity.square().mean((1, 2, 3, 4)).sqrt(),
            "velocity_max": velocity.abs().amax((1, 2, 3, 4)),
            "divergence_relative": (
                divergence.square().mean((1, 2, 3)).sqrt()
                / scale.clamp_min(1e-8)
            ),
            "pressure_rhs_mean_correction": correction.abs().flatten(1).amax(1),
        }


## Required original physics preflight


In [ ]:
@torch.no_grad()
def physics_preflight():
    transform_probe = torch.randn(
        1, 8, 3, 3, device=device, dtype=torch.complex64)
    dct_roundtrip = float(
        (idct2_complex(dct2_complex(transform_probe))
         - transform_probe).abs().max())
    dst_roundtrip = float(
        (idst4_complex(dst4_complex(transform_probe))
         - transform_probe).abs().max())
    transform_n = transform_probe.shape[1]
    spacing = 1.0 / transform_n
    nn_laplacian = torch.empty_like(transform_probe)
    nn_laplacian[:, 1:-1] = (
        transform_probe[:, :-2] - 2 * transform_probe[:, 1:-1]
        + transform_probe[:, 2:]) / spacing ** 2
    nn_laplacian[:, 0] = (
        transform_probe[:, 1] - transform_probe[:, 0]) / spacing ** 2
    nn_laplacian[:, -1] = (
        transform_probe[:, -2] - transform_probe[:, -1]) / spacing ** 2
    nn_lambda = (
        -4.0
        * torch.sin(
            math.pi * torch.arange(
                transform_n, device=device) / (2 * transform_n)
        ).square()
        / spacing ** 2
    ).reshape(1, transform_n, 1, 1)
    nn_eigen_residual = float(
        torch.linalg.vector_norm(
            dct2_complex(nn_laplacian)
            - nn_lambda * dct2_complex(transform_probe))
        / torch.linalg.vector_norm(
            dct2_complex(nn_laplacian)).clamp_min(1e-12))

    dn_laplacian = torch.empty_like(transform_probe)
    dn_laplacian[:, 1:-1] = (
        transform_probe[:, :-2] - 2 * transform_probe[:, 1:-1]
        + transform_probe[:, 2:]) / spacing ** 2
    dn_laplacian[:, 0] = (
        transform_probe[:, 1] - 3 * transform_probe[:, 0]) / spacing ** 2
    dn_laplacian[:, -1] = (
        transform_probe[:, -2] - transform_probe[:, -1]) / spacing ** 2
    dn_lambda = (
        -4.0
        * torch.sin(
            math.pi * (
                torch.arange(transform_n, device=device) + 0.5
            ) / (2 * transform_n)).square()
        / spacing ** 2
    ).reshape(1, transform_n, 1, 1)
    dn_eigen_residual = float(
        torch.linalg.vector_norm(
            dst4_complex(dn_laplacian)
            - dn_lambda * dst4_complex(transform_probe))
        / torch.linalg.vector_norm(
            dst4_complex(dn_laplacian)).clamp_min(1e-12))
    n = min(PDE.reference_n, 32)
    simulator = PorousConvectionReference3D(PDE, n=n)
    z = (torch.arange(n, device=device) + 0.5) / n
    hydrostatic = torch.erfc(
        z / (2 * math.sqrt(PDE.diffusivity * PDE.initial_time))
    )[None, None, :, None, None].expand(1, 1, n, n, n).contiguous()
    hydro_velocity, hydro_faces, _ = simulator.green.face_and_cell_velocity(hydrostatic)
    hydro_ratio = float(
        hydro_velocity.square().mean().sqrt()
        / hydrostatic.square().mean().sqrt().clamp_min(1e-12))

    initial = simulator.initial_condition([123])
    velocity, faces, correction = simulator.green.face_and_cell_velocity(initial)
    divergence = simulator.green.face_divergence(faces)
    divergence_relative = float(
        divergence.square().mean().sqrt()
        / (velocity.square().mean().sqrt() / (1 / n)).clamp_min(1e-12))
    wall_normal = float(torch.maximum(faces[2][:, 0].abs().amax(),
                                      faces[2][:, -1].abs().amax()))
    before_mass = float(initial.mean())
    before_flux = float(simulator.top_flux(initial))
    after, info = simulator.advance_frame(initial, min(PDE.frame_dt, 0.04))
    after_mass = float(after.mean())
    mass_delta = after_mass - before_mass
    integrated_flux_per_depth = float(
        info["integrated_top_flux"].mean() / PDE.depth)
    mass_balance_relative = abs(
        mass_delta - integrated_flux_per_depth
    ) / max(abs(integrated_flux_per_depth), 1e-10)
    mass_rate = (after_mass - before_mass) / min(PDE.frame_dt, 0.04)
    range_min, range_max = float(after.min()), float(after.max())
    result = {
        "dct_roundtrip_max": dct_roundtrip,
        "dst_roundtrip_max": dst_roundtrip,
        "neumann_dct_eigen_residual": nn_eigen_residual,
        "dirichlet_neumann_dst_eigen_residual": dn_eigen_residual,
        "hydrostatic_carrier_relative": hydro_ratio,
        "face_divergence_relative": divergence_relative,
        "wall_normal_face_max": wall_normal,
        "pressure_rhs_mean_correction": float(correction.abs().max()),
        "initial_top_flux": before_flux,
        "one_frame_mass_rate": mass_rate,
        "one_frame_integrated_top_flux_per_depth": integrated_flux_per_depth,
        "one_frame_mass_flux_balance_relative": mass_balance_relative,
        "one_frame_min_max": [range_min, range_max],
        "one_frame_substeps": info["substeps"],
    }
    print(json.dumps(result, indent=2))
    assert dct_roundtrip < 2e-5 and dst_roundtrip < 2e-5
    assert nn_eigen_residual < 2e-5 and dn_eigen_residual < 2e-5
    assert hydro_ratio < 2e-4
    assert divergence_relative < 2e-4
    assert wall_normal < 1e-7
    assert result["pressure_rhs_mean_correction"] < 1e-5
    assert before_flux > 0 and mass_rate > 0
    assert mass_balance_relative < 0.15
    assert range_min > -2e-3 and range_max < 1.002
    del simulator
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


PHYSICS_PREFLIGHT = physics_preflight()


## Generate/resume signed per-trajectory arrays and diagnostics


In [ ]:
SPLIT_COUNTS = {"train": PDE.n_train, "val": PDE.n_val, "test": PDE.n_test}
SPLIT_TRANSITIONS = {
    "train": PDE.train_transitions,
    "val": PDE.val_transitions,
    "test": PDE.test_transitions,
}
SPLIT_SEED_OFFSETS = {"train": 100_000, "val": 200_000, "test": 300_000}


def trajectory_path(split, trajectory):
    return DATA_ROOT / split / f"trajectory_{trajectory:04d}.npy"


def diagnostic_path(split, trajectory):
    return DATA_ROOT / split / f"trajectory_{trajectory:04d}_diagnostics.csv"


def expected_shape(split):
    return (
        SPLIT_TRANSITIONS[split] + 1,
        1,
        PDE.model_n,
        PDE.model_n,
        PDE.model_n,
    )


def valid_existing_trajectory(split, trajectory):
    path = trajectory_path(split, trajectory)
    if not path.exists() or not diagnostic_path(split, trajectory).exists():
        return False
    try:
        array = np.load(path, mmap_mode="r")
        diagnostics = pd.read_csv(diagnostic_path(split, trajectory))
        return (
            tuple(array.shape) == expected_shape(split)
            and str(array.dtype) == STORAGE_NUMPY_DTYPE
            and len(diagnostics) == SPLIT_TRANSITIONS[split] + 1
        )
    except Exception:
        return False


def downsample_reference(state):
    factor = PDE.downsample_factor
    if factor == 1:
        return state
    return F.avg_pool3d(state, kernel_size=factor, stride=factor)


@torch.no_grad()
def generate_batch(split, trajectory_ids):
    simulator = PorousConvectionReference3D(PDE)
    continuum_seeds = [
        SPLIT_SEED_OFFSETS[split] + int(index) for index in trajectory_ids
    ]
    state = simulator.initial_condition(continuum_seeds)
    sampled_initial_times = [
        simulator.sampled_initial_time(seed) for seed in continuum_seeds]
    warmup_elapsed = 0.0
    while warmup_elapsed < PDE.warmup_time - 1e-12:
        dt = min(PDE.frame_dt, PDE.warmup_time - warmup_elapsed)
        state, _ = simulator.advance_frame(state, dt)
        warmup_elapsed += dt

    frames = [[] for _ in trajectory_ids]
    diagnostic_rows = [[] for _ in trajectory_ids]
    initial_masses = state.float().mean((-3, -2, -1))[:, 0].clone()
    cumulative_flux = torch.zeros(
        len(trajectory_ids), device=state.device, dtype=torch.float32)
    transitions = SPLIT_TRANSITIONS[split]
    for frame in tqdm(
            range(transitions + 1),
            desc=f"simulate {split} trajectories {trajectory_ids[0]}..{trajectory_ids[-1]}"):
        stored = (
            downsample_reference(state).float().cpu().numpy()
            .astype(STORAGE_NUMPY_DTYPE, copy=False))
        diagnostics = simulator.diagnostics(state)
        for local_id in range(len(trajectory_ids)):
            frames[local_id].append(stored[local_id])
            diagnostic_rows[local_id].append({
                "frame": frame,
                "time_after_warmup": frame * PDE.frame_dt,
                "sampled_initial_diffusive_time": sampled_initial_times[local_id],
                "absolute_convective_time": (
                    sampled_initial_times[local_id]
                    + PDE.warmup_time + frame * PDE.frame_dt),
                "absolute_diffusive_time": (
                    PDE.rayleigh_darcy
                    * (sampled_initial_times[local_id]
                       + PDE.warmup_time + frame * PDE.frame_dt)),
                "cumulative_top_flux": float(cumulative_flux[local_id]),
                "mass_balance_residual": float(
                    diagnostics["mass"][local_id]
                    - initial_masses[local_id]
                    - cumulative_flux[local_id] / PDE.depth),
                **{key: float(value[local_id]) for key, value in diagnostics.items()},
            })
        if frame < transitions:
            state, step_info = simulator.advance_frame(state)
            cumulative_flux += step_info["integrated_top_flux"]
            qmin, qmax = float(state.min()), float(state.max())
            if qmin < -5e-3 or qmax > 1.005:
                raise FloatingPointError(
                    f"Reference solver lost boundedness: min={qmin}, max={qmax}")

    for local_id, trajectory in enumerate(trajectory_ids):
        directory = DATA_ROOT / split
        directory.mkdir(parents=True, exist_ok=True)
        array = np.stack(frames[local_id]).astype(
            STORAGE_NUMPY_DTYPE, copy=False)
        target = trajectory_path(split, trajectory)
        temporary = target.with_suffix(".npy.partial")
        with temporary.open("wb") as handle:
            np.save(handle, array)
        os.replace(temporary, target)
        pd.DataFrame(diagnostic_rows[local_id]).to_csv(
            diagnostic_path(split, trajectory), index=False)
    del simulator, state
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


DATASET_MANIFEST_PATH = DATA_ROOT / "dataset_manifest.json"
EXPECTED_DATASET_MANIFEST = {
    "data_signature": DATA_SIGNATURE,
    "dataset_version": DATASET_VERSION,
    "pde": asdict(PDE),
    "splits": SPLIT_COUNTS,
    "transitions": SPLIT_TRANSITIONS,
    "storage": (
        f"{STORAGE_NUMPY_DTYPE} cell-centred concentration; generated at "
        "reference_n and volume-averaged"),
}
if DATASET_MANIFEST_PATH.exists():
    existing_manifest = json.loads(DATASET_MANIFEST_PATH.read_text())
    if existing_manifest != EXPECTED_DATASET_MANIFEST:
        raise RuntimeError(
            "Dataset manifest mismatch. The signed data directory must not be reused.")
else:
    atomic_json(DATASET_MANIFEST_PATH, EXPECTED_DATASET_MANIFEST)

if RUN_DATA_GENERATION:
    for split, count in SPLIT_COUNTS.items():
        missing = [
            trajectory for trajectory in range(count)
            if not valid_existing_trajectory(split, trajectory)
        ]
        if not missing:
            print(f"{split}: all {count} trajectories already exist")
            continue
        for start in range(0, len(missing), PDE.generation_batch):
            generate_batch(split, missing[start:start + PDE.generation_batch])

for split, count in SPLIT_COUNTS.items():
    missing = [
        trajectory for trajectory in range(count)
        if not valid_existing_trajectory(split, trajectory)
    ]
    if missing:
        raise FileNotFoundError(
            f"{split} is incomplete; missing/invalid trajectories: {missing[:10]}")
    print(split, {
        "trajectories": count,
        "shape": expected_shape(split),
        "disk_GB": sum(
            trajectory_path(split, i).stat().st_size for i in range(count)) / 1e9,
    })

print("DATA_SIGNATURE", DATA_SIGNATURE)


## Training-only normalization, fixed starts, and original dataset-readiness audit


In [ ]:
class PorousWindowStore:
    def __init__(self, split, horizon):
        self.split = split
        self.horizon = int(horizon)
        self.files = [
            trajectory_path(split, trajectory)
            for trajectory in range(SPLIT_COUNTS[split])
        ]
        self.lengths = [
            np.load(path, mmap_mode="r").shape[0] for path in self.files
        ]
        self.references = []
        for trajectory, length in enumerate(self.lengths):
            for start in range(length - self.horizon):
                self.references.append((trajectory, start))
        if not self.references:
            raise RuntimeError(f"No {split} windows of horizon {horizon}")

    def read(self, reference):
        trajectory, start = reference
        array = np.load(self.files[trajectory], mmap_mode="r")
        block = np.asarray(
            array[start:start + self.horizon + 1], dtype=np.float32).copy()
        return torch.from_numpy(block)


def fixed_starts(split, horizon):
    starts = []
    stride = max(1, (SPLIT_TRANSITIONS[split] - horizon) // 3)
    candidates = (0, stride, 2 * stride, 3 * stride)
    for trajectory in range(SPLIT_COUNTS[split]):
        maximum = SPLIT_TRANSITIONS[split] - horizon
        starts.append((trajectory, min(candidates[trajectory % 4], maximum)))
    return starts


class FixedStartPorousDataset(Dataset):
    def __init__(self, split, horizon):
        self.store = PorousWindowStore(split, horizon)
        self.references = fixed_starts(split, horizon)

    def __len__(self):
        return len(self.references)

    def __getitem__(self, item):
        return self.store.read(self.references[item])


def compute_training_statistics():
    count = total = 0
    state_sum = state_sq = delta_sum = delta_sq = 0.0
    for trajectory in range(PDE.n_train):
        array = np.asarray(
            np.load(trajectory_path("train", trajectory), mmap_mode="r"),
            dtype=np.float64)
        values = array[:, 0]
        deltas = np.diff(values, axis=0)
        state_sum += values.sum()
        state_sq += np.square(values).sum()
        delta_sum += deltas.sum()
        delta_sq += np.square(deltas).sum()
        count += values.size
        total += deltas.size
    state_mean = state_sum / count
    state_std = math.sqrt(max(state_sq / count - state_mean ** 2, 1e-12))
    delta_mean = delta_sum / total
    delta_std = math.sqrt(max(delta_sq / total - delta_mean ** 2, 1e-14))
    return {
        "state_mean": [state_mean],
        "state_std": [state_std],
        "delta_mean": [delta_mean],
        "delta_std": [delta_std],
        "data_signature": DATA_SIGNATURE,
    }


STATS_PATH = DATA_ROOT / "train_statistics.json"


if STATS_PATH.exists():
    STATS = json.loads(STATS_PATH.read_text())
    if STATS.get("data_signature") != DATA_SIGNATURE:
        raise RuntimeError("Stale train statistics; remove or regenerate the dataset")
else:
    STATS = compute_training_statistics()
    atomic_json(STATS_PATH, STATS)


print("Statistics below normalize inputs, outputs, and loss; they are used in training.")


print(json.dumps(STATS, indent=2))


assert STATS["state_std"][0] > 1e-6 and STATS["delta_std"][0] > 1e-8


FIXED_STARTS_PATH = DATA_ROOT / "fixed_evaluation_starts.json"


atomic_json(FIXED_STARTS_PATH, {
    "validation": fixed_starts("val", RUN.val_rollout),
    "test": fixed_starts("test", RUN.test_rollout),
    "data_signature": DATA_SIGNATURE,
})


print("Fixed starts:", json.loads(FIXED_STARTS_PATH.read_text()))


@torch.no_grad()
def dataset_readiness_audit():
    dataset = FixedStartPorousDataset("test", RUN.test_rollout)
    green = SlabDarcyGreen3D(
        PDE.model_n, (PDE.depth, PDE.length_y, PDE.length_x)).to(device)
    rows = []
    for index in range(len(dataset)):
        sequence = dataset[index].to(device)
        initial = sequence[0:1]
        final = sequence[-1:]
        carrier = green(initial)
        displacement_cells = (
            carrier.square().sum(1).sqrt()
            * PDE.frame_dt * PDE.model_n / PDE.depth
        )
        persistence = float(
            torch.linalg.vector_norm(final - initial)
            / torch.linalg.vector_norm(final).clamp_min(1e-8))
        horizontal_std = float(
            final[:, 0].std((-2, -1)).mean())
        rows.append({
            "trajectory": index,
            "persistence_H_relative_l2": persistence,
            "horizontal_std_final": horizontal_std,
            "route_q50_cells": float(torch.quantile(displacement_cells, 0.50)),
            "route_q90_cells": float(torch.quantile(displacement_cells, 0.90)),
            "mass_growth": float(final.mean() - initial.mean()),
        })
    frame = pd.DataFrame(rows)
    summary = frame.mean(numeric_only=True).to_dict()
    print(frame)
    print("Dataset readiness mean:", json.dumps(summary, indent=2))
    if PROFILE == "paper":
        failures = []
        if summary["persistence_H_relative_l2"] < 0.03:
            failures.append("H-step task is too close to persistence")
        if summary["horizontal_std_final"] < 0.005:
            failures.append("fingering/horizontal anomaly is too weak")
        if summary["route_q90_cells"] < 0.10:
            failures.append("saved frame spacing produces a weak Green route")
        if failures:
            message = "Paper dataset readiness failed: " + "; ".join(failures)
            if ALLOW_WEAK_DATASET:
                warnings.warn(message)
            else:
                raise RuntimeError(
                    message + ". Change the physical sampling design, not this gate.")
    frame.to_csv(DATA_ROOT / "dataset_readiness.csv", index=False)
    return summary


DATASET_READINESS = dataset_readiness_audit()
